# Tutorial 9: Recurrent GNNs

In this tutorial we will implement an approximation of the Graph Neural Network Model (without enforcing contraction map) and analyze the GatedGraph Convolution of Pytorch Geometric.

In [2]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

# Install compatible CPU versions of PyTorch and Torchvision

!pip install -q --index-url https://download.pytorch.org/whl/cpu torch==2.12.0 torchvision==0.27.0

2.14.1+cpu


In [5]:
# Reinstall matching CPU versions of PyTorch and Torchvision

!pip uninstall -y torch torchvision torchaudio torch-geometric pyg-lib torch-scatter torch-sparse

!pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cpu torch==2.12.0 torchvision==0.27.0

Found existing installation: torch 2.12.0+cpu
Uninstalling torch-2.12.0+cpu:
  Successfully uninstalled torch-2.12.0+cpu
Found existing installation: torchvision 0.27.0+cpu
Uninstalling torchvision-0.27.0+cpu:
  Successfully uninstalled torchvision-0.27.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Found existing installation: torch-geometric 2.9.0
Uninstalling torch-geometric-2.9.0:
  Successfully uninstalled torch-geometric-2.9.0
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 158.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 164.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision]


In [3]:
# Force reinstall the correct CPU versions

!pip install --force-reinstall --no-cache-dir --index-url https://download.pytorch.org/whl/cpu torch==2.12.0 torchvision==0.27.0

Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 244.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 203.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 225.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 125.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 199.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 176.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 189.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 209.8 MB/s  0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_

In [2]:
# Install PyTorch Geometric in the current Colab environment

import torch
print("Current PyTorch:", torch.__version__)

%pip install torch-geometric==2.8.1

Current PyTorch: 2.12.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.7 MB/s  0:00:00


In [1]:
# Test the PyTorch Geometric installation

import torch
import torch_geometric

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("PyG import successful!")

PyTorch: 2.12.0+cpu
PyG: 2.8.1
PyG import successful!


In [2]:
import os.path as osp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
import torch_geometric
from torch_geometric.datasets import Planetoid, TUDataset
from torch_geometric.data import DataLoader
from torch_geometric.nn.inits import uniform
from torch.nn import Parameter as Param
from torch import Tensor
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = "cpu"
from torch_geometric.nn.conv import MessagePassing

In [3]:
dataset = 'Cora'
transform = T.Compose([T.TargetIndegree(),
])
path = osp.join('data', dataset)
dataset = Planetoid(path, dataset, transform=transform)
data = dataset[0]

Processing...
Done!


In [4]:
dataset = 'Cora'
path = osp.join('data', dataset)
dataset = Planetoid(path, dataset, transform=T.NormalizeFeatures())
data = dataset[0]
data = data.to(device)

### Graph Neural Network Model

<img src="https://github.com/AntonioLonga/PytorchGeometricTutorial/blob/main/Tutorial9/transition.png?raw=1" width="500">

<img src="https://github.com/AntonioLonga/PytorchGeometricTutorial/blob/main/Tutorial9/output.png?raw=1" width="500">

The MLP class is used to instantiate the transition and output functions as simple feed forard networks

In [5]:
class MLP(nn.Module):
    def __init__(self, input_dim, hid_dims, out_dim):
        super(MLP, self).__init__()

        self.mlp = nn.Sequential()
        dims = [input_dim] + hid_dims + [out_dim]
        for i in range(len(dims)-1):
            self.mlp.add_module('lay_{}'.format(i),nn.Linear(in_features=dims[i], out_features=dims[i+1]))
            if i+2 < len(dims):
                self.mlp.add_module('act_{}'.format(i), nn.Tanh())
    def reset_parameters(self):
        for i, l in enumerate(self.mlp):
            if type(l) == nn.Linear:
                nn.init.xavier_normal_(l.weight)

    def forward(self, x):
        return self.mlp(x)



The GNNM calss puts together the state propagations and the readout of the nodes' states.

In [6]:
class GNNM(MessagePassing):
    def __init__(self, n_nodes, out_channels, features_dim, hid_dims, num_layers = 50, eps=1e-3, aggr = 'add',
                 bias = True, **kwargs):
        super(GNNM, self).__init__(aggr=aggr, **kwargs)

        self.node_states = Param(torch.zeros((n_nodes, features_dim)), requires_grad=False)
        self.out_channels = out_channels
        self.eps = eps
        self.num_layers = num_layers

        self.transition = MLP(features_dim, hid_dims, features_dim)
        self.readout = MLP(features_dim, hid_dims, out_channels)

        self.reset_parameters()
        print(self.transition)
        print(self.readout)

    def reset_parameters(self):
        self.transition.reset_parameters()
        self.readout.reset_parameters()

    def forward(self):
        edge_index = data.edge_index
        edge_weight = data.edge_attr
        node_states = self.node_states
        for i in range(self.num_layers):
            m = self.propagate(edge_index, x=node_states, edge_weight=edge_weight,
                               size=None)
            new_states = self.transition(m)
            with torch.no_grad():
                distance = torch.norm(new_states - node_states, dim=1)
                convergence = distance < self.eps
            node_states = new_states
            if convergence.all():
                break

        out = self.readout(node_states)

        return F.log_softmax(out, dim=-1)

    def message(self, x_j, edge_weight):
        return x_j if edge_weight is None else edge_weight.view(-1, 1) * x_j

    def message_and_aggregate(self, adj_t, x) :
        return matmul(adj_t, x, reduce=self.aggr)

    def __repr__(self):
        return '{}({}, num_layers={})'.format(self.__class__.__name__,
                                              self.out_channels,
                                              self.num_layers)

In [7]:
model = GNNM(data.num_nodes, dataset.num_classes, 32, [64,64,64,64,64], eps=0.01).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()


test_dataset = dataset[:len(dataset) // 10]
train_dataset = dataset[len(dataset) // 10:]
test_loader = DataLoader(test_dataset)
train_loader = DataLoader(train_dataset)

def train():
    model.train()
    optimizer.zero_grad()
    loss_fn(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()


def test():
    model.eval()
    logits, accs = model(), []
    for _, mask in data('train_mask', 'val_mask', 'test_mask'):
        pred = logits[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


for epoch in range(1, 51):
    train()
    accs = test()
    train_acc = accs[0]
    val_acc = accs[1]
    test_acc = accs[2]
    print('Epoch: {:03d}, Train Acc: {:.5f}, '
          'Val Acc: {:.5f}, Test Acc: {:.5f}'.format(epoch, train_acc,
                                                       val_acc, test_acc))

MLP(
  (mlp): Sequential(
    (lay_0): Linear(in_features=32, out_features=64, bias=True)
    (act_0): Tanh()
    (lay_1): Linear(in_features=64, out_features=64, bias=True)
    (act_1): Tanh()
    (lay_2): Linear(in_features=64, out_features=64, bias=True)
    (act_2): Tanh()
    (lay_3): Linear(in_features=64, out_features=64, bias=True)
    (act_3): Tanh()
    (lay_4): Linear(in_features=64, out_features=64, bias=True)
    (act_4): Tanh()
    (lay_5): Linear(in_features=64, out_features=32, bias=True)
  )
)
MLP(
  (mlp): Sequential(
    (lay_0): Linear(in_features=32, out_features=64, bias=True)
    (act_0): Tanh()
    (lay_1): Linear(in_features=64, out_features=64, bias=True)
    (act_1): Tanh()
    (lay_2): Linear(in_features=64, out_features=64, bias=True)
    (act_2): Tanh()
    (lay_3): Linear(in_features=64, out_features=64, bias=True)
    (act_3): Tanh()
    (lay_4): Linear(in_features=64, out_features=64, bias=True)
    (act_4): Tanh()
    (lay_5): Linear(in_features=64, ou

/tmp/ipykernel_4453/3431952659.py:8: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_dataset)
/tmp/ipykernel_4453/3431952659.py:9: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dataset)


Epoch: 001, Train Acc: 0.18571, Val Acc: 0.21000, Test Acc: 0.18800
Epoch: 002, Train Acc: 0.15000, Val Acc: 0.16600, Test Acc: 0.14400
Epoch: 003, Train Acc: 0.19286, Val Acc: 0.13400, Test Acc: 0.11300
Epoch: 004, Train Acc: 0.15714, Val Acc: 0.12800, Test Acc: 0.09400
Epoch: 005, Train Acc: 0.10714, Val Acc: 0.12600, Test Acc: 0.12600
Epoch: 006, Train Acc: 0.17857, Val Acc: 0.21200, Test Acc: 0.20400
Epoch: 007, Train Acc: 0.17857, Val Acc: 0.20600, Test Acc: 0.20000
Epoch: 008, Train Acc: 0.17857, Val Acc: 0.19000, Test Acc: 0.19700
Epoch: 009, Train Acc: 0.22857, Val Acc: 0.17400, Test Acc: 0.17100
Epoch: 010, Train Acc: 0.21429, Val Acc: 0.12800, Test Acc: 0.11000
Epoch: 011, Train Acc: 0.20714, Val Acc: 0.14000, Test Acc: 0.11500
Epoch: 012, Train Acc: 0.20714, Val Acc: 0.15400, Test Acc: 0.12700
Epoch: 013, Train Acc: 0.20714, Val Acc: 0.16800, Test Acc: 0.13000
Epoch: 014, Train Acc: 0.21429, Val Acc: 0.15800, Test Acc: 0.12600
Epoch: 015, Train Acc: 0.20000, Val Acc: 0.17200

### Gated Graph Neural Network

In [8]:
class GatedGraphConv(MessagePassing):

    def __init__(self, out_channels, num_layers, aggr = 'add',
                 bias = True, **kwargs):
        super(GatedGraphConv, self).__init__(aggr=aggr, **kwargs)

        self.out_channels = out_channels
        self.num_layers = num_layers

        self.weight = Param(Tensor(num_layers, out_channels, out_channels))
        self.rnn = torch.nn.GRUCell(out_channels, out_channels, bias=bias)

        self.reset_parameters()

    def reset_parameters(self):
        uniform(self.out_channels, self.weight)
        self.rnn.reset_parameters()

    def forward(self, data):
        """"""
        x = data.x
        edge_index = data.edge_index
        edge_weight = data.edge_attr
        if x.size(-1) > self.out_channels:
            raise ValueError('The number of input channels is not allowed to '
                             'be larger than the number of output channels')

        if x.size(-1) < self.out_channels:
            zero = x.new_zeros(x.size(0), self.out_channels - x.size(-1))
            x = torch.cat([x, zero], dim=1)

        for i in range(self.num_layers):
            m = torch.matmul(x, self.weight[i])
            m = self.propagate(edge_index, x=m, edge_weight=edge_weight,
                               size=None)
            x = self.rnn(m, x)

        return x

    def message(self, x_j, edge_weight):
        return x_j if edge_weight is None else edge_weight.view(-1, 1) * x_j

    def message_and_aggregate(self, adj_t, x):
        return matmul(adj_t, x, reduce=self.aggr)

    def __repr__(self):
        return '{}({}, num_layers={})'.format(self.__class__.__name__,
                                              self.out_channels,
                                              self.num_layers)

class GGNN(torch.nn.Module):
    def __init__(self):
        super(GGNN, self).__init__()

        self.conv = GatedGraphConv(1433, 3)
        self.mlp = MLP(1433, [32,32,32], dataset.num_classes)

    def forward(self):
        x = self.conv(data)
        x = self.mlp(x)
        return F.log_softmax(x, dim=-1)


In [9]:
model = GGNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()


test_dataset = dataset[:len(dataset) // 10]
train_dataset = dataset[len(dataset) // 10:]
test_loader = DataLoader(test_dataset)
train_loader = DataLoader(train_dataset)

def train():
    model.train()
    optimizer.zero_grad()
    loss_fn(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()


def test():
    model.eval()
    logits, accs = model(), []
    for _, mask in data('train_mask', 'val_mask', 'test_mask'):
        pred = logits[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


for epoch in range(1, 51):
    train()
    accs = test()
    train_acc = accs[0]
    val_acc = accs[1]
    test_acc = accs[2]
    print('Epoch: {:03d}, Train Acc: {:.5f}, '
          'Val Acc: {:.5f}, Test Acc: {:.5f}'.format(epoch, train_acc,
                                                       val_acc, test_acc))

/tmp/ipykernel_4453/96977798.py:8: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_dataset)
/tmp/ipykernel_4453/96977798.py:9: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dataset)


Epoch: 001, Train Acc: 0.15000, Val Acc: 0.13600, Test Acc: 0.13100
Epoch: 002, Train Acc: 0.17857, Val Acc: 0.25400, Test Acc: 0.23500
Epoch: 003, Train Acc: 0.18571, Val Acc: 0.13400, Test Acc: 0.13800
Epoch: 004, Train Acc: 0.29286, Val Acc: 0.22000, Test Acc: 0.22500
Epoch: 005, Train Acc: 0.36429, Val Acc: 0.34800, Test Acc: 0.33200
Epoch: 006, Train Acc: 0.40000, Val Acc: 0.36800, Test Acc: 0.34100
Epoch: 007, Train Acc: 0.40714, Val Acc: 0.37800, Test Acc: 0.35300
Epoch: 008, Train Acc: 0.42857, Val Acc: 0.41800, Test Acc: 0.39000
Epoch: 009, Train Acc: 0.54286, Val Acc: 0.48800, Test Acc: 0.46900
Epoch: 010, Train Acc: 0.55714, Val Acc: 0.51800, Test Acc: 0.50300
Epoch: 011, Train Acc: 0.50714, Val Acc: 0.48400, Test Acc: 0.47100
Epoch: 012, Train Acc: 0.46429, Val Acc: 0.46400, Test Acc: 0.46000
Epoch: 013, Train Acc: 0.50000, Val Acc: 0.48600, Test Acc: 0.49000
Epoch: 014, Train Acc: 0.57857, Val Acc: 0.51600, Test Acc: 0.51200
Epoch: 015, Train Acc: 0.60714, Val Acc: 0.54000